# RAG

LLMs have been trained on large amounts of data, but they are not trained on your data.
Retrieval Augmented Generation (RAG) addresses this limitation by providing your data directly to the LLM.

In RAG, your data is loaded and indexed. The user query is then encoded and compared to each index, filtering your data to the most relevant text chunks. The selected chunks are then added to the prompt, along side you query, which then go to the LLM to generate the final response.

<img src="media/basic_rag.png" width="700">

The standard setup of a RAG system follows the following steps:

<img src="media/rag_stages.png" width="700">

### Install Required Packages

In [1]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-together

### Imports

In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, get_response_synthesizer, Settings, StorageContext, load_index_from_storage
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.llms.together import TogetherLLM
from llama_index.core.node_parser import SimpleNodeParser, SentenceSplitter

import torch

/anaconda/envs/MLCat/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the Documents

Loading the documents from which we want to get answers. There are three papers in the `papers` folder:  
- "Strategies for shortening tuberculosis therapy"
- "A multimodal vision foundation model for clinical dermatology"
- "Individual variations in glycemic responses to carbohydrates and underlying metabolic physiology"

In the documents variable, each page from all of the papers is stored.

In [3]:
documents = SimpleDirectoryReader("papers").load_data()

## Configure Embedding Model and LLM

Define the model that will be used to generate the document chunks embeddings and the query embedding.

We will be using the encoder model `"sentence-transformers/multi-qa-mpnet-base-dot-v1"`.

In [4]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/multi-qa-mpnet-base-dot-v1",
    device="cpu"
)

Define the model that will be used to generate the response of the query.

We will be using the LLM `"meta-llama/Llama-3.2-3B-Instruct-Turbo"`.

In [5]:
api_key="ca8fa3a23e31d53a3af2be301fa66f04ae095fe1ae6e66da480e1eb03c845888"

In [6]:
Settings.llm = TogetherLLM(
    model="meta-llama/Llama-3.2-3B-Instruct-Turbo", api_key=api_key
)

## Split and Index Sources

Create vector index. When you use `from_documents`, the documents are split into chunks which are then indexed.

In [7]:
# Create the text splitter
text_splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50,
    paragraph_separator="\n\n",
    secondary_chunking_regex="[.!?]"
)

# Split the text into nodes
nodes = text_splitter.get_nodes_from_documents(documents)

View the first 2 nodes:

In [8]:
for i, node in enumerate(nodes[:2]):
    print(f"\nNode {i + 1}:\n{node.text}")


Node 1:
Nature Medicine
nature medicine
https://doi.org/10.1038/s41591-025-03747-y
Article
 e-mail: zongyuan.ge@monash.edu
A list of authors and their affiliations appears at the end of the paper
A multimodal vision foundation model for 
clinical dermatology
Diagnosing and treating skin diseases require advanced visual skills across 
domains and the ability to synthesize information from multiple imaging 
modalities. While current deep learning models excel at specific tasks 
such as skin cancer diagnosis from dermoscopic images, they struggle 
to meet the complex, multimodal requirements of clinical practice. Here 
we introduce PanDerm, a multimodal dermatology foundation model 
pretrained through self-supervised learning on over 2 million real-world 
skin disease images from 11 clinical institutions across 4 imaging modalities. 
We evaluated PanDerm on 28 diverse benchmarks, including skin cancer 
screening, risk stratification, differential diagnosis of common and rare skin 
condit

In [9]:
# # create index
# index = VectorStoreIndex.from_documents(nodes)

# # store index locally
# index.storage_context.persist(persist_dir="stored_index")

# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="stored_index")

# load index
index = load_index_from_storage(storage_context)

Loading llama_index.core.storage.kvstore.simple_kvstore from stored_index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from stored_index/index_store.json.


## Querying and Retrieving

Configure the retriever, which will fetch the most relevant chunks.

In [10]:
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=3,
)

Configure the response synthesizer (LLM).

In [11]:
response_synthesizer = get_response_synthesizer()

Assemble the query engine.

In [12]:
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.55)],
)

## Get Retrived Chunks and Generate Response

Function that uses the LLM to generate the response from the retrieved chunks and prints them.

In [13]:
def query_with_sources(query_text, print_chunks=False):
    """Query the engine and display retrieved chunks"""
    print(f"\nQuery: {query_text}")
    print("=" * 50)
    
    # Get response
    response = query_engine.query(query_text)
    
    if print_chunks:
        # Display retrieved chunks
        print("\n\nRetrieved Chunks:")
        print("-" * 20)

        if hasattr(response, 'source_nodes'):
            for i, node in enumerate(response.source_nodes):
                print(f"\nChunk {i+1}:")
                print(f"Score: {node.score:.4f}")
                print(f"Source: {node.node.metadata.get('file_name', 'Unknown')}")
                print(f"Content: {node.node.text[:1000]}...")  # First 500 chars
                print("-" * 40)
        else:
            print("No source nodes available")
        
    # Display the response
    print("\nResponse:")
    print("-" * 20)
    print(response)
    
    return response

# Exercises

## LLM vs RAG

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h5>📝 Exercise 1</h5>
  <p>Compare the answers obtained directly from the LLM with the answer obtained using RAG.</p>
</div>

In [14]:
query_text = "What is the effect of fluoroquinolone on the duration of a treatment?"

The answer to the question is present on the `tuberculosis_therapy.pdf` paper.

<img src="media/rag_example.png" width="700">

### Using Only an LLM

Get the answer using the LLM directly.

In [15]:
llm = TogetherLLM(
    model="meta-llama/Llama-3.2-3B-Instruct-Turbo",
    api_key=api_key,
)

response = llm.complete(query_text)
print(response.text)

Fluoroquinolones are a class of antibiotics that can affect the duration of treatment for various bacterial infections. The effect of fluoroquinolones on treatment duration can vary depending on the specific infection, the type of fluoroquinolone used, and the individual patient's response to the medication.

 Generally, fluoroquinolones can:

1. Shorten treatment duration: Fluoroquinolones are often effective against a wide range of bacterial infections, which can lead to a shorter treatment duration compared to other antibiotics.
2. Reduce treatment duration for certain infections: Fluoroquinolones are often used to treat infections such as urinary tract infections (UTIs), respiratory tract infections, and skin infections. In these cases, fluoroquinolones can reduce the treatment duration compared to other antibiotics.
3. Increase treatment duration for certain infections: However, fluoroquinolones may require longer treatment durations for certain infections, such as bone and joint 

### Using RAG

Get the answer using RAG, so that the engine retrieves the relevant chunks and uses them to inform the generation of the answer.

In [16]:
response = query_with_sources(query_text, print_chunks=True)


Query: What is the effect of fluoroquinolone on the duration of a treatment?


Retrieved Chunks:
--------------------

Chunk 1:
Score: 0.6102
Source: tuberculosis_therapy.pdf
Content: More recent clinical trials support this proposition. Three large 
phase 3 trials examined the possibility that adding a fluoroquinolone 
to standard treatment could reduce the duration of therapy to 4 
months38–40. All three trials failed to demonstrate non-inferiority of 
the shortened regimen when comparing relapses across all patients. A 
reanalysis of the individual patient data subsequently showed that the 
4-month regimens were non-inferior in a large proportion of patients 
with minimal disease but were clearly inferior in patients with more 
extensive cavitary disease2.
Patients with HTT disease tend to have specific pathological 
features (cavitary disease in particular), and multiple studies have 
shown that such patients have a higher risk of disease relapse even 
formidable challenge of deal

The RAG system was able to select the correct paper and sections!

## Similarity Threshold Tuning

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h5>📝 Exercise 2</h5>
  <p>Experiment with the following similarity_cutoffs: 0.55 and 0.75. What does this do? What is the impact of this change?</p>
</div>

Pay close attention to:
- Number of chunks retrieved
- Subjective relevance quality
- Response completeness

In [17]:
query_text = "What is PanDerm, and how is it different from previous dermatology AI models?"

<img src="media/panderm_rag.png" width="700">

In [18]:
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.55)], # <------- CHANGE HERE
)

In [19]:
response = query_with_sources(query_text, print_chunks=True)


Query: What is PanDerm, and how is it different from previous dermatology AI models?


Retrieved Chunks:
--------------------

Chunk 1:
Score: 0.7547
Source: foundation_model_dermatology.pdf
Content: Nature Medicine
Article https://doi.org/10.1038/s41591-025-03747-y
+0.25; Fig. 6g) and top 3 accuracy (generalist group, +16.5%; derma-
tology group, +10.3%; Fig. 6h). Analysis by condition classes showed 
consistent improvements across inflammatory, neoplastic and other 
categories (P < 0.05; Fig. 6i,j), with inflammatory conditions show-
ing the largest gains (+0.36 in top 1 diagnostic scores, +14.2% in top 3 
accuracy). Furthermore, when used independently, PanDerm achieved 
higher diagnostic accuracy than both unassisted readers (top 1 scores: 
3.6 versus 2.83; P < 0.001) and human–AI collaboration (top 1 scores: 
3.6 versus 3.08; P < 0.001). Further details on the setup, methodology, 
reader statistics and datasets of all three reader studies are provided 
in Methods, Extended Data F